<div style="background-color:#e8f4f8; border: 2px solid #3399cc; padding: 15px; border-radius: 8px;">

<h1><b>Step-by-step Guidance - Running the NEETML Tool</b></h1>

<span style="display: inline-block; font-size: 20px; color: #2f4f5f; background-color: rgba(255,255,255,0.55); padding: 5px 10px; border-radius: 999px; margin: 2px 0 16px 0;">
  <span style="letter-spacing: 0.4px; text-transform: uppercase; font-size: 20px; font-weight: 700; color: #3399cc;">Author</span>
  <span style="color:#9aa; margin: 0 7px;">•</span>
  <span style="font-weight: 600;">Yanhua Xu</span>
</span>

<div style="font-size: 40px; font-weight: 500; color: #444; border-top: 1px solid #999; padding-top: 8px; margin-top: 6px;">
Step 2 - Feature Engineering
</div>

<!-- # **Step 2 - Feature Engineering** -->
<!-- <h1><b> Step 2 - Feature Engineering </b></h1> -->

This notebook extends the prepared dataset by introducing new variables and transforming existing ones to enhance model performance and interpretability.

It includes:  
- Merging external datasets (e.g., IMD and IoD, School Performance)  
- Deriving features from raw data
- Aggregating data with conflicting information
- Deriving composite indicators and encoded variables  
- Performing numerical transformations or scaling  
- Constructing model-ready features 

</div>

# 1. <a id='toc1_'></a>[Pre-requesite](#toc0_)

## 1.1. <a id='toc1_1_'></a>[Python and required packages](#toc0_)

In [ ]:
import sys
from pathlib import Path
import datetime
import pandas as pd
import yaml
from rich import print
import numpy as np
from neetml.feature_engineering import FeatEngineer

import warnings
warnings.filterwarnings("ignore")

# 2. <a id='toc2_'></a>[Initialisation](#toc0_)

In [ ]:
overwrite = False

feat_engineer = FeatEngineer(overwrite=overwrite)

# find the path for the merged dataset the produced by the raw data preparation step
from neetml.raw_prep import RawDataCurator
merged_dir = RawDataCurator().get_data_path(keys='merged').Path[0]
merged_path = list(Path(merged_dir).glob('*.parquet'))[0] # Assuming there's only one parquet file in the directory

print(f"Path to the merged dataset: {merged_path}")

output_filename = merged_path.name
feat_engineer.set_output_filename(output_filename)

# 3. <a id='toc3_'></a>[Manual fix column name inconsistency issues (temporary solution; optional if data is already consistent)](#toc0_)

::: {.callout-warning}

Double-check is needed for this step

:::

Some columns in attendance data have inconsistent names. We can check for such inconsistencies and standardise them.

Some columns still cannot be standardised due to the following reasons:

- 'attendance_educational_count': not sure if it is 'attendance_educational_visit_v' or 'attendance_count'. 'attendance_educational_count' only exists in data collected in the second patch. It seems 'attendance_educational_visit_v' but 'attendance_visit_v_count' seems more likely to be 'attendance_educational_visit_v' as 'attendance_educational_count' is not the same as 'attendance_educational_visit_v' in terms of range of values. The range of 'attendance_educational_count' seems the same as 'attendance_count' but 'attendance_count' exists in all data. So we don't do any mapping for 'attendance_educational_count'.

- 'attendance_family_holiday' and 'attendance_agreed_family_holiday': they only exist in 2022-2023_Y11' cohort. We map 'attendance_agreed_family_holiday' to 'attendance_agreed_hols_h_count' and map 'attendance_family_holiday' to 'attendance_unagreed_hols_g_count' based on the range of values, but we are not sure if they are the same. 

- 'attendance_traveller': it only contains 0 and 1, while 'attendance_traveller_t_count' contains a wider range of values. We map 'attendance_traveller' to 'attendance_traveller_t_count' but we are not sure if they are the same. 

- 'attendance_*_illness': we map it to 'attendance_illness_i_count' based on the name similarity, but we are not sure if they are the same as the range of values is different.

- 'attendance_exceptional_count': it is almost identical to 'attendance_exceptional_circumstances' but the only differences occur where 'attendance_excep_circ_y_count' has a non-zero value and 'attendance_exceptional_count' has 0. We map 'attendance_exceptional_count' to 'attendance_excep_circ_y_count'

- following columns only exist in 2022-2023_Y11' cohort, we are not sure if they are the same as any columns in other cohorts, so we don't do any mapping for them:
  - attendance_late:  attendance_late_after_reg_u_count or attendance_late_before_reg_l_count ?
  - attendance_registration_closed:   # ?
  - attendance_not_covered: attendance_other_unauth_o_count ?
  - attendance_hash:  # ?
  - attendance_dash: # ?
  - attendance_pupil_not_yet_on_roll:  # ?
  - attendance_illness_confirmed_coronavirus: # ?
  - attendance_non_compulsory_school_age:  # ?

In [ ]:
from neetml.utils.misc import find_comp_pairs, apply_col_map, plot_col_coverage

# create the dir for the curated dataset to be saved
curated_dir = merged_path.parent.parent / (merged_path.parent.name + '_curated')
curated_dir.mkdir(exist_ok=True)
# curated_path = curated_dir / merged_path.with_name(merged_path.stem + '_curated.parquet').name
curated_path = curated_dir / save_name

if curated_path.exists() and not overwrite:
    df = pd.read_parquet(curated_path) # or set input_data_path to curated_path
    print(f'The curated dataset has been loaded from {curated_path}.')
else:
    print(f"Curated dataset does not exist at {curated_path}. Processing the merged dataset to create the curated dataset...")
    df = pd.read_parquet(merged_path)

    # 'attendance_absence_count' is new variable since 2022-2023_Y7, so fill the missing values of 'attendance_absence_count' with the sum of 'attendance_authorised_count' and 'attendance_unauthorised_count' if they are the same for the non-missing values, to keep the most complete data for 'attendance_absence_count'
    if not (df[['attendance_authorised_count', 'attendance_unauthorised_count']].sum(axis=1, skipna=True) == df['attendance_absence_count']).eq(False).any():
        # if 'attendance_absence_count' is the same as the sum of 'attendance_authorised_count' and 'attendance_unauthorised_count', then fill the missing values
        df['attendance_absence_count'].fillna(df['attendance_unauthorised_absence'] + df['attendance_authorised_absence'], inplace=True)
        df['attendance_absence_count'].fillna(df['attendance_authorised'] + df['attendance_unauthorised'], inplace=True)

    # 'attendance_new_possible' only exists in 2022-2023_attendance_Y11, it seems contains the updated 
    # It seems 'attendance_authorised' (abscence count) + 'attendance_unauthorised' (abscence count) + 'attendance_sessions_attended' = 'attendance_new_possible'
    if not (df[['attendance_authorised', 'attendance_unauthorised', 'attendance_sessions_attended']].sum(axis=1, skipna=True) == df['attendance_new_possible']).eq(False).any():
        df['attendance_new_possible'].fillna(df['attendance_possible_sessions'], inplace=True) # 'attendance_new_possible' maybe the updated version of 'attendance_possible_sessions' in 2022-2023_Y11, so fill the missing values with 'attendance_possible_sessions' to keep the most complete data for 'attendance_possible_sessions'
        df.drop(columns=['attendance_possible_sessions'], inplace=True)
        df.rename(columns={'attendance_new_possible': 'attendance_possible_sessions'}, inplace=True)

    
    print(f"\n Scanning data for complementary, name-similar & dtype-check column pairs...")

    cols = [
        col for col in df.columns
        if col != 'stud_id'
        and not col.startswith(('_', 'ks2', 'ks4'))
    ]

    _, minor_to_major_map = find_comp_pairs(df=df, cols=cols, thres=0.8) 
    
    save_path = curated_dir / "col_map.yaml"
    with open(save_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            minor_to_major_map,
            f,
            sort_keys=True,
            allow_unicode=True,
            default_flow_style=False
        )
    
    curated_yaml_path = save_path.stem + "_curated.yaml"
    
    if (curated_dir / curated_yaml_path).exists():
        col_map = yaml.load((curated_dir / curated_yaml_path).open(), Loader=yaml.FullLoader)
    else:
        col_map = yaml.load((curated_dir / save_path.name).open(), Loader=yaml.FullLoader)
    
    df_curated = apply_col_map(df, col_map, drop_from_col=True, copy=True, verbose=True)
    
    print('Done. The complementary, name-similar & dtype-check column pairs have been mapped.')

    # Some columns need to be recalculated or split
    # 'attendance_laestab' contains la code and establishment code, which can be split into two separate columns 'attendance_la' and filled the missing value of 'attendance_estab_no' 
    df_curated['attendance_la'] = df_curated['attendance_laestab'].str[:3]
    df_curated['attendance_estab_no'].fillna(df_curated['attendance_laestab'].str[3:], inplace=True)

    # do the same for ks2 and ks4 laestab columns
    df_curated['ks2_la'] = df_curated['ks2_laestab'].str[:3]
    df_curated['ks4_la'] = df_curated['ks4_laestab'].str[:3]
    
    # 'attendance_possible_sessions' may not be accurated. It seems it should be number of attended sessions + unauthorised and authorised absences, so recalculate it to keep the most complete data for 'attendance_possible_sessions'
    df_curated.drop(columns=['attendance_possible_sessions'], inplace=True)
    df_curated['attendance_possible_sessions'] = df_curated['attendance_count'] + df_curated['attendance_unauthorised_count'] + df_curated['attendance_authorised_count']
        
    # drop some columns that only exists in one cohort but cannot be mapped to any other columns in other cohorts
    df_curated.drop(columns=['attendance_late', 'attendance_registration_closed', 'attendance_not_covered', 'attendance_hash', 'attendance_pupil_not_yet_on_roll', 'attendance_illness_confirmed_coronavirus', 'attendance_non_compulsory_school_age', 'attendance_dash'], inplace=True, errors='ignore')
    
    # also drop 'attendance_possible_sessions_less_covid_absence' as it only exists in three cohorts
    df_curated.drop(columns=['attendance_possible_sessions_less_covid_absence'], inplace=True, errors='ignore')
    
    # percentage columns only exist in new data, so fill the missing values with the percentage calculated from the count columns
    pct_cols = [col for col in df_curated.columns if '%' in col]
    for pct_col in pct_cols:
        count_col = pct_col.replace('%', 'count')
        if count_col in df_curated.columns:
            den = df_curated["attendance_possible_sessions"]
            num = df_curated[count_col]
            mask = den.notna() & (den > 0) & num.notna()
            
            df_curated[pct_col] = np.nan
            df_curated.loc[mask, pct_col] = num[mask] / den[mask]
        else:
            print(f"Warning: The count column {count_col} for {pct_col} is not found. The percentage column will be filled with NaN.")
            df_curated[pct_col] = pd.NA
    
    for col in ['attendance', 'census']:
        plot_col_coverage(
            df_curated,
            prefix=col,
            figsize=(25, 15),
            verbose=True,
            save_excel=True,
            output_file=str(curated_dir / f'{col}_cols_presence.jpg')
        )
    
    df_curated.to_parquet(curated_path, index=False)
    
    df = df_curated.copy()

check current data path that used for feature engineering and make sure it is correct.

In [ ]:
feat_engineer.get_data_path(keys='all')

In [ ]:
# feat_engineer.set_input_data_path(curated_path)

# feat_engineer.get_data_path(keys='all')

# 4. <a id='toc4_'></a>[Link public datasets](#toc0_)

## 4.1. <a id='toc4_1_'></a>[Link school performance data](#toc0_)

### 4.1.1. <a id='toc4_1_1_'></a>[Process and add school performance features](#toc0_)

#### 4.1.1.1. <a id='toc4_1_1_1_'></a>[Download school performance data and merge them into single file per year](#toc0_)

We download historical school performance data from [Compare School Performance](https://www.compare-school-performance.service.gov.uk/download-data?)

<!-- https://www.compare-school-performance.service.gov.uk/download-data?currentstep=region&downloadYear=2016-2017&regiontype=la&la=380 -->
<!-- https://www.compare-school-performance.service.gov.uk/download-data?currentstep=region&downloadYear=2017-2018&regiontype=la&la=0 -->

By default we only download school information data

No data can be found in 2019-2020 academic year.

*NOTE: all percentage columns in the source files are converted to percentage format (e.g., 85.5 for 85.5%)*

In [ ]:
# # la = 380 # Bradford
# la = 0 # All of England (as some children studied outside of bradford)
# fmt = 'xlsx'
# current_year = datetime.date.today().year

# fetch_kwargs={
#     "la_code": la,
#     "academic_start_year": 2013,
#     "academic_end_year": current_year,
#     "file_format": fmt,
# }

# feat_engineer.download_school_data(**fetch_kwargs)

# df_school = feat_engineer.build_school_perf_data()

### 4.1.2. <a id='toc4_1_2_'></a>[Append school info data to linked dataset](#toc0_)

In [ ]:
df_att_susp = feat_engineer.link_school_perf_data(input_data=df, school_ref=["att", "susp"], output_name="link_perf_1.parquet") # by default the input_year_col = '_acad_year'
df_ks2 = feat_engineer.link_school_perf_data(input_data=df_att_susp, school_ref=["ks2"], input_year_col='ks2_examyear_re', output_name="link_perf_2.parquet")
df = feat_engineer.link_school_perf_data(input_data=df_ks2, school_ref=['ks4'], input_year_col='_cohort_y11_ay', output_name="link_perf_3.parquet")

del df_att_susp, df_ks2

## 4.2. <a id='toc4_2_'></a>[Link IoD and IMD data](#toc0_)

In [ ]:
# Optional pre-processing steps
# For records before Year 11, only NCCIS code was appended, so NCCIS postcode is missing for those records.
# To keep NCCIS postcode as complete as possible, fill the missing postcode values in pre-Year-11 records
# with the corresponding NCCIS postcode recorded at Year 11.

if 'nccis_postcode' in df.columns and 'nccis_code' in df.columns:
    # get each person's Year 11 postcode
    y11_postcode = (
        df.loc[(df["_year_group"] == '11') & (df["_phase"] == "pre-spring"), ["stud_id", "nccis_postcode"]]
        .dropna(subset=["nccis_postcode"])
        .drop_duplicates()
        .set_index("stud_id")["nccis_postcode"]
    )

    # map Year 11 postcode back to all records
    df["_y11_postcode"] = df["stud_id"].map(y11_postcode)

    # fill missing postcode ONLY for records before Year 11
    mask = (df["_year_group"].astype(int) < 11) & (df["nccis_postcode"].isna())

    df.loc[mask, "nccis_postcode"] = df.loc[mask, "_y11_postcode"]
    
    df.drop(columns="_y11_postcode", inplace=True)

In [ ]:
df = feat_engineer.add_iod(input_data=df, col_pd='nccis_postcode', output_name="link_perf_iod.parquet")

# 5. <a id='toc5_'></a>[Derive features](#toc0_)

## 5.1. <a id='toc5_1_'></a>[Pre-processing steps for deriving ethnicity/gender/language/school features](#toc0_)

### 5.1.1. <a id='toc5_1_1_'></a>[Ethnicity](#toc0_)

The `census_ethnicity` column contains a mixture of (extended and main) ethnicity codes (e.g. `WBRI, AIND`) and about 20\% main/sub category labels (e.g. `Pakistani, Asian Or Asian British`), all of which need to be standardised before use.

Process steps:
1. curate the `census_ethnicity` with `suspPermExcl_ethnicity`
2. map all curated codes to DfE main category labels
3. replace following labels that cannot be found in DfE main category labels:
    `White British And Irish` -> `White`,
    `Mixed` -> `Mixed/Dual Background`,
    `Other Ethnic` -> `Any Other Ethnic Group`
    `Not On Jan Census` -> `Unknown`

*Reference of DfE ethnicity codes: https://www.gov.uk/guidance/alternative-provision-ap-census/codes

In [ ]:
from neetml.utils.constants import DER_COL_PREFIX

# the current census_ethnicity column contains a mixture of ethnicity codes (e.g. `WBRI, AIND`) and labels (e.g. `White other, Pakistani, Asian Or Asian British`)
# where it seems the labels are from the approved extended categories and main categories, so we use suspPermExcl_ethnicity to curate the census_ethnicity column

df["_ethnicity_raw"] = df["suspPermExcl_ethnicity"].astype("object").fillna(df["census_ethnicity"].astype("object"))    

df = df.drop(columns=['suspPermExcl_ethnicity', 'census_ethnicity']).drop_duplicates()

df["_ethnicity_raw"].replace(
    {
        'White British And Irish': 'White',
        'Mixed': 'Mixed/Dual Background',
        'Other Ethnic': 'Any Other Ethnic Group',
        'Not On Jan Census': "Unknown"
    }, inplace=True)

### 5.1.2. <a id='toc5_1_2_'></a>[Gender](#toc0_)

<blockquote style="background-color: #FFFFE0; padding: 15px; margin: 0; border-left: 5px solid #FFD700;">
TBD: explanation
</blockquote>

In [ ]:
gender_cols = [col for col in df.columns if any(x in col.lower() for x in ["gender", "sex"]) and col != 'ks2_gpsexp' and 'type' not in col]
df[gender_cols] = df[gender_cols].replace({"Female": "F", "Male": "M", "Unknown": pd.NA})

print(gender_cols)

### 5.1.3. <a id='toc5_1_3_'></a>[Language](#toc0_)

<blockquote style="background-color: #FFFFE0; padding: 15px; margin: 0; border-left: 5px solid #FFD700;">
TBD: explanation
</blockquote>

In [ ]:
language_cols = [col for col in df.columns if any(x in col.lower() for x in ["language"])]
print(language_cols)

df["_language_raw"] = df["census_language"].astype("object").fillna(df["suspPermExcl_language"].astype("object"))    

df = df.drop(columns=['census_language', 'suspPermExcl_language'], errors='ignore').drop_duplicates()

### 5.1.4. <a id='toc5_1_4_'></a>[School](#toc0_)

<blockquote style="background-color: #FFFFE0; padding: 15px; margin: 0; border-left: 5px solid #FFD700;">
TBD: explanation
</blockquote>

In [ ]:
df = df.drop(columns=['nccis_establishment_type'], errors='ignore') # only two rows have non-noll values in 'nccis_establishment_type'

df['_year_group'] = df['_year_group'].astype("Int64")
df['_phase_num'] = df['_phase'].astype(object).fillna(0).replace('pre-spring', 1).replace('post-spring', 2)

In [ ]:
from neetml.utils.code_mappings import ks_nftype_map

# map ks2/ks4 school type codes to categories
nftype_cols = [col for col in df.columns if 'nftype' in col]
for c in nftype_cols:
    new_cols = c.replace('nftype', 'school_type')
    df[new_cols] = df[c].astype("Int64").map(ks_nftype_map)

In [ ]:
urn_cols = [col for col in df.columns if 'urn' in col and not col.startswith('ks2') and 'ac' not in col]
pcd_cols = [col for col in df.columns if 'postcode' in col and 'ks2' not in col and 'nccis' not in col]
gender_type_cols = [col for col in df.columns if 'gender_type' in col and 'ks2' not in col]

exclude_cols = {"ks4_nftype", "ks4_new_type", "ks4_newer_type", "nccis_establishment_type"} # not considered them for now

type_cols = [
    col for col in df.columns
    if "school_type" in col
    and "ks2" not in col
    and col not in exclude_cols
]

## 5.2. <a id='toc5_2_'></a>[Derived features](#toc0_)

**Table of contents**<a id='toc0_'></a>    
1. [Pre-requesite](#toc1_)    
1.1. [Python and required packages](#toc1_1_)    
2. [Initialisation](#toc2_)    
3. [Manual fix column name inconsistency issues (temporary solution; optional if data is already consistent)](#toc3_)    
4. [Link public datasets](#toc4_)    
4.1. [Link school performance data](#toc4_1_)    
4.1.1. [Process and add school performance features](#toc4_1_1_)    
4.1.1.1. [Download school performance data and merge them into single file per year](#toc4_1_1_1_)    
4.1.2. [Append school info data to linked dataset](#toc4_1_2_)    
4.2. [Link IoD and IMD data](#toc4_2_)    
5. [Derive features](#toc5_)    
5.1. [Pre-processing steps for deriving ethnicity/gender/language/school features](#toc5_1_)    
5.1.1. [Ethnicity](#toc5_1_1_)    
5.1.2. [Gender](#toc5_1_2_)    
5.1.3. [Language](#toc5_1_3_)    
5.1.4. [School](#toc5_1_4_)    
5.2. [Derived features](#toc5_2_)    
6. [Data aggregation / conflicts consolidation](#toc6_)    
7. [Drop columns with same values](#toc7_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=true
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<blockquote style="background-color: #FFFFE0; padding: 15px; margin: 0; border-left: 5px solid #FFD700;">
TBD: explanation
</blockquote>

Generate high-level ethnicity groups and conflict indicators

This step creates clearer ethnicity information from the original ethnicity column.

The raw data may only contain one ethnicity value, such as a DfE extended ethnicity code or category. To make this easier to use, this step maps the original value to the standard ethnicity levels shown in the ethnicity reference table.

The reference table includes:

| Reference field | Meaning |
|---|---|
| `DfE Extended Code` | the detailed ethnicity code, for example `WBRI` |
| `Approved Extended Categories` | the detailed ethnicity category, for example `White - British` |
| `DfE Main Code` | the broader DfE ethnicity code, for example `WBRI` |
| `Sub- Category` | the broader sub-category, for example `White - British` |
| `Main Category` | the highest-level ethnicity group, for example `White` |

After this step, new ethnicity-related columns are added to the dataset. These give different levels of ethnicity detail, from the most detailed code to the broadest main category. By default, the derived variables are created with the prefix `der_`.

For example, a value such as `WBRI` can be mapped to:

| Derived field | Example value |
|---|---|
| `der_ethnicity_ext_code` | `WBRI` |
| `der_ethnicity_ext_cat` | `White - British` |
| `der_ethnicity_sub_cat` | `White - British` |
| `der_ethnicity_main_code` | `WBRI` |
| `der_ethnicity_main_cat` | `White` |

**Multi-ethnicity check**

This step can also check whether the same student has more than one substantive ethnicity recorded.

By default, this check is done at the `main_cat` level. This means the function checks whether each student has more than one main ethnicity category, such as `White`, `Asian`, `Black`, `Mixed`, or `Other`.

The check level can be changed depending on how detailed the check should be:

| `check_level` | What it checks |
|---|---|
| `main_cat` | checks conflict at the broadest main category level; this is the default |
| `main_code` | checks conflict using the DfE main code |
| `sub_cat` | checks conflict using the ethnicity sub-category |
| `ext_cat` | checks conflict using the approved extended category |
| `ext_code` | checks conflict using the detailed DfE extended code |
| `False` | skips the multi-ethnicity check |

If conflicting ethnicity values are found for the same student at the selected check level, the conflict is flagged and the curated ethnicity value is left empty to avoid using misleading information.

Non-substantive values such as `Refused`, `Unknown`, or `Not obtained` are excluded from the conflict check.

**Variables created**

| New variable | What it contains |
|---|---|
| `der_ethnicity_ext_code` | the detailed DfE extended ethnicity code |
| `der_ethnicity_ext_cat` | the approved extended ethnicity category |
| `der_ethnicity_sub_cat` | the ethnicity sub-category |
| `der_ethnicity_main_code` | the DfE main ethnicity code |
| `der_ethnicity_main_cat` | the broad main ethnicity category |
| `der_ethnicity_conflict_n` | the number of different substantive ethnicity values recorded for the same student at the selected check level |
| `der_ethnicity_conflict_flag` | whether the student has more than one substantive ethnicity value at the selected check level |
| `der_ethnicity_curated` | the curated ethnicity value based on the selected check level; this is left empty if there is a conflict |

Generate curated gender variables

We found some inconsistencies in gender information across different data sources. We can derive gender conflict indicators to identify records with conflicting gender information. in the current dataset, we only have very few records with conflicting gender information, so we just use `census_gender` as key gender variable and backfill missing values with `ks4_gender`.

Generate features regarding changes of estab status

In [ ]:
df_new = feat_engineer.derive_features(
    input_data=df,
    input_path=curated_path,
    derive_ethnicity=True,
    derive_gender=True,
    derive_school=True,
    derive_language=True,
    ethnicity_kwargs={
        "ethnic_col": "_ethnicity_raw",
        "check_level": "main_cat",
        # "time_cols": ['_year_group', '_phase_num'], 
    },
    gender_kwargs={
        "gender_cols": gender_cols,
        # "time_cols": ['_year_group', '_phase_num'], 
    },
    school_kwargs={
        "urn_cols": urn_cols,
        "pcd_cols": pcd_cols,
        "type_cols": type_cols,
        "gender_type_cols": gender_type_cols,
        "time_cols": ['_year_group', '_phase_num'], 
        "clean_type": True,
        "features": ("hist", "main", "state_chg"),
        "state_threshold": 2
    },
    language_kwargs={
        "language_col": "_language_raw",
        "time_cols": ['_year_group', '_phase_num'], 
        "features": ("hist", "main", "n_uniq"),
    }
)

df = df_new.copy()

array_cols = [
    col for col in df.columns
    if df[col].map(lambda x: isinstance(x, np.ndarray)).any()
]

if array_cols:

    print(array_cols)

    for col in array_cols:
        df[col] = df[col].map(
            lambda x: tuple(x.tolist()) if isinstance(x, np.ndarray) else x
        )

df = df.drop(columns=["_ethnicity_raw", "_language_raw"] + gender_cols, errors='ignore').drop_duplicates()

# 6. <a id='toc6_'></a>[Data aggregation / conflicts consolidation](#toc0_)

The goal of this step is to make sure each category has the correct output grain, for example:

```text
Census:                 one row per student-year / census point
Attendance:             one row per student-year group
KS2:                    one row per student
KS4:                    one row per student
Suspension/Exclusion:   one row per student-year group
```

General rules for resolving conflicts:

| Category | Resolution Strategy |
|---|---|
| Census | Select row by enrolment status priority |
| KS2 | Select base row, then fill missing values |
| KS4 | Select most complete/base row, then fill missing values |
| Attendance | Sum counts and recalculate percentages |
| Suspension/Exclusion | Keep latest event and count events |

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Details (click to expand to)
</summary>


**Detecting Conflicts**

Conflicts are detected by checking whether more than one row exists for the expected grouping key.

**Census**

Census conflicts mainly come from multiple enrolment statuses.

The resolver selects one main row using this priority:

```text
M > C > S > other/missing
```

Where:

```text
M = current main registration
C = current single registration
S = current subsidiary registration
```

**KS2**

KS2 duplicates are handled by selecting a base row, then filling missing values from other duplicate rows.

Base row selection uses:

```text
completeness > distance from expected KS2 year and entry date of corresponding records
```

If date of birth is available, expected KS2 year is estimated as:

```text
Jan-Aug birth: birth year + 11
Sep-Dec birth: birth year + 12
```

Backfill rule:

```text
If base row value is missing:
    fill from another duplicate row

If base row already has a value:
    keep base row value
```

Only pupil-level attainment/background columns should be backfilled. School IDs, year fields, candidate IDs, and metadata should not be mixed across rows.


**KS4**

KS4 uses similar logic to KS2.

If a KS4 year column exists:

```text
completeness > year distance
```

If no year column exists:

```text
completeness > original row order
```

After selecting the base row, other duplicated rows can fill missing values in selected KS4 result/background columns.

The resolver does not overwrite non-missing base values.

**Attendance**

Attendance records are aggregated, not resolved by keeping one row.

Rules:

```text
count/session columns -> sum
percentage columns -> recalculate from summed counts
other columns -> keep first/latest according to configuration
```

Percentages should not be averaged directly.

Example:

```text
attendance_absence_% =
    attendance_absence_count / attendance_possible_sessions * 100
```

This avoids incorrect percentages when duplicated rows have different numbers of possible sessions.

**Suspension and Permanent Exclusion**

Suspension/exclusion data is event-level, so multiple rows can be valid.

The simplified resolver does:

```text
1. Keep the latest event row within each group
2. Add an event count for that group
```

the output keeps one latest event per student-year group and adds:

```text
suspPermExcl_event_count
```

</details>


In [ ]:
df = df.drop(columns=['der_ethnicity_main_cat', 'der_ethnicity_main_code', 'der_ethnicity_sub_cat', 'der_ethnicity_ext_cat', 'der_ethnicity_ext_code'], errors='ignore').drop_duplicates()

df_new = feat_engineer.resolve_conflicts(
  input_data=df,
  overwrite=False,
  group_keys=['stud_id', '_cohort_y11_ay', '_year_group', '_acad_year', '_phase_num'],
  use_conflict_ids=True,
)

# 7. <a id='toc7_'></a>[Drop columns with same values](#toc0_)

In [ ]:
col_hashes = df_new.apply(lambda s: pd.util.hash_pandas_object(s, index=False).sum())

keep_cols = col_hashes[~col_hashes.duplicated()].index

df_new_dedup = df_new[keep_cols]

df_new_dedup.to_parquet(merged_dir.parent / output_filename, index=False)

In [ ]:
df_new_dedup

In [ ]:
# double check if there are any columns that have the same values across all rows

same_cols = []

cols = sorted(df_new.columns)

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df_new[cols[i]].equals(df_new[cols[j]]):
            same_cols.append((cols[i], cols[j]))

same_cols